In [1]:
from config import DATA_DIR
import pandas as pd
from pathlib import Path


In [2]:
DATA_PATH = DATA_DIR / "raw" / "Protest_pg_ready.csv"
df = pd.read_csv(DATA_PATH, sep=";")


In [3]:
import pandas as pd
response_cols = [
    c for c in df.columns
    if "Primary State Response to protests" in c
]

response_cols


['Primary State Response to protests [Response 1]',
 'Primary State Response to protests [Response 2?¾]',
 'Primary State Response to protests [Response 3]',
 'Primary State Response to protests [Response 4]',
 'Primary State Response to protests [Response 5]',
 'Primary State Response to protests [Response 6]',
 'Primary State Response to protests [Response 7]']

In [4]:
import pandas as pd

all_responses = pd.Series(dtype="object")

for col in response_cols:
    all_responses = pd.concat([all_responses, df[col]])

print(all_responses.shape)


(104811,)


In [5]:
clean_responses = (
    all_responses
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
)

print(clean_responses.shape)


(17663,)


In [6]:
unique_responses = sorted(clean_responses.unique())

print(f"Number of unique responses: {len(unique_responses)}")
for r in unique_responses:
    print(r)


Number of unique responses: 8
.
accomodation
arrests
beatings
crowd dispersal
ignore
killings
shootings


In [7]:
import pandas as pd

# 1) Find state response columns
response_cols = [
    c for c in df.columns
    if "Primary State Response to protests" in c
]

# 2) Define keyword sets
repression_keywords = {
    "arrests",
    "beatings",
    "crowd dispersal",
    "shootings",
    "killings"
}

accommodation_keywords = {
    "accomodation"
}

# 3) Initialize target features
df["target_repression"] = 0
df["target_accommodation"] = 0

# 4) Process each response column
for col in response_cols:
    col_clean = (
        df[col]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    # Boolean masks (DO NOT filter the Series itself)
    valid_mask = col_clean != "."
    repression_mask = col_clean.isin(repression_keywords) & valid_mask
    accommodation_mask = col_clean.isin(accommodation_keywords) & valid_mask

    # Assign values using aligned boolean masks
    df.loc[repression_mask, "target_repression"] = 1
    df.loc[accommodation_mask, "target_accommodation"] = 1

# 5) Sanity check
print("Sanity check:")
print(df[["target_repression", "target_accommodation"]].value_counts())


Sanity check:
target_repression  target_accommodation
0                  0                       8744
1                  0                       4883
0                  1                        902
1                  1                        444
Name: count, dtype: int64


In [8]:
import pandas as pd

# 1) Create binary target from numeric column
df["violent"] = (
    pd.to_numeric(df["Protester Violence"], errors="coerce")
    .fillna(0)
    .astype(int)
)

# 2) Sanity checks
print("Violence distribution:")
print(df["violent"].value_counts())

print("\nViolence rate:")
print(df["violent"].mean().round(3))


Violence distribution:
violent
0    11406
1     3567
Name: count, dtype: int64

Violence rate:
0.238


In [9]:
import re
import numpy as np

def clean_participants(x):
    if pd.isna(x):
        return -1

    x = str(x).lower().strip()

    # obvious text-based quantities
    if any(k in x for k in ["dozen", "dozens"]):
        return 1          # ~50–199
    if any(k in x for k in ["busload", "busloads"]):
        return 2          # ~200–999
    if any(k in x for k in ["hundred", "hundreds", "hunderates", "100s"]):
        return 2
    if any(k in x for k in ["thousand", "thousands", "1000s"]):
        return 3

    # ranges like 100-400
    match = re.findall(r"\d+", x)
    if len(match) >= 2:
        avg = np.mean([int(match[0]), int(match[1])])
    elif len(match) == 1:
        avg = int(match[0])
    else:
        return -1

    # binning
    if avg < 50:
        return 0
    elif avg < 200:
        return 1
    elif avg < 1000:
        return 2
    elif avg < 5000:
        return 3
    else:
        return 4


# apply
df["participants_bin"] = df["Reported participation in protest"].apply(clean_participants)

# sanity check
df["participants_bin"].value_counts(dropna=False).sort_index()


participants_bin
-1    1775
 0      58
 1    4347
 2    2878
 3    3393
 4    2522
Name: count, dtype: int64

In [10]:
import pandas as pd
import numpy as np

# -----------------------------
# 1) num_demands
# -----------------------------
demand_cols = [
    "Primary Protester Demands [Demand 1]",
    "Primary Protester Demands [Demand 2]",
    "Primary Protester Demands [Demand 3]",
    "Primary Protester Demands [Demand 4]"
]

df["num_demands"] = df[demand_cols].notna().sum(axis=1)


# -----------------------------
# 2) decade (from Year)
# -----------------------------
df["decade"] = (df["Year"] // 10) * 10


# -----------------------------
# 3) Final sanity check
# -----------------------------
print("Feature check:")
print(df[["num_demands", "decade"]].head())
print("\nMissing values:")
print(df[["num_demands", "decade"]].isna().sum())


Feature check:
   num_demands  decade
0            2    1990
1            1    1990
2            1    1990
3            1    1990
4            1    1990

Missing values:
num_demands    0
decade         0
dtype: int64


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.ensemble import GradientBoostingClassifier
import pandas as pd
import numpy as np


# -----------------------------
# 1) Feature matrix
# -----------------------------
features = [
    "participants_bin",
    "target_repression",
    "target_accommodation",
    "num_demands",
    "decade"
]

X = df[features]
y = df["violent"]


# -----------------------------
# 2) Train / Test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42
)


# -----------------------------
# 3) Optimized Gradient Boosting
# -----------------------------
model = GradientBoostingClassifier(
    n_estimators=500,        # بیشتر → یادگیری الگوهای نادر
    learning_rate=0.03,     # نرم‌تر، پایدارتر
    max_depth=2,            # جلوگیری از overfitting
    min_samples_leaf=30,    # حساس‌تر به کلاس اقلیت
    subsample=0.8,
    random_state=42
)

model.fit(X_train, y_train)


# -----------------------------
# 4) Probabilities
# -----------------------------
y_proba = model.predict_proba(X_test)[:, 1]


# -----------------------------
# 5) Threshold tuned for Recall
# -----------------------------
THRESHOLD = 0.35   # ✅ مناسب EWS
y_pred = (y_proba >= THRESHOLD).astype(int)


print(f"\nROC-AUC: {roc_auc_score(y_test, y_proba):.3f}\n")
print(f"Classification report (threshold = {THRESHOLD}):")
print(classification_report(y_test, y_pred))


# -----------------------------
# 6) Feature importance
# -----------------------------
fi = (
    pd.DataFrame({
        "feature": features,
        "importance": model.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

print("\nFeature importance:")
print(fi)



ROC-AUC: 0.848

Classification report (threshold = 0.35):
              precision    recall  f1-score   support

           0       0.94      0.80      0.86      2852
           1       0.56      0.82      0.67       892

    accuracy                           0.80      3744
   macro avg       0.75      0.81      0.76      3744
weighted avg       0.85      0.80      0.81      3744


Feature importance:
                feature  importance
1     target_repression    0.950031
0      participants_bin    0.021938
4                decade    0.010395
2  target_accommodation    0.009200
3           num_demands    0.008437
